In [ ]:
####Importing all the libraries needed for the Machine learning project

#### For Data manipulation and wrangling:
import pandas as pandas ### for data handling and manipulation
import numpy as np      ### for  working with numbers and arrays
import seaborn as sns   ### for data visualization
import matplotlib.pyplot as plt  ### for data visualization

#### Preprocessing and Pipelines:
from sklearn.pipeline import Pipeline  ### for automating the preprocessing steps to allow sequential run
from sklearn.compose import ColumnTransformer ## to scale the numeric columns and encode categorical columns
from sklearn.preprocessing import OrdinalEncoder ### to convert categorical columns into numbers where there is order criteria
from sklearn.preprocessing import StandardScaler   ### scales numerical columns to have a mean = 0 and std 1
from sklearn.model_selection import train_test_split  ### for data splitting into test and train
from sklearn.impute import SimpleImputer     #### for handling missing values
from imblearn.under_sampling import RandomUnderSampler    ### handles imblanced data by randomly reducing the size of majority class

#### For the Supervised Machine learning models:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

### For Model Evaluation:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
#### To load the data set:
genetic_data = pd.read_csv(r"C:\Users\cassa\Downloads\Genetic_disorder.csv")
genetic_data

In [ ]:
####Data type check:

genetic_data.shape


##### The data set has five thousand eight hundered and eighty fiver rows and fourty-five columns.


In [ ]:
###D Data Type check for each columns:

genetic_data.info()

##### The result of the data above shows that each column is in the correct data type and there are no missing values. 

In [ ]:
#### Data preview

genetic_data.head()

In [ ]:
#### Data Quality Check:

genetic_data[genetic_data.duplicated()]


##### There are no duplicates rows in the data set

In [ ]:
### to duplicate the dataset to avoid any changes on the original data set
df_genetic = genetic_data.copy()
df_genetic

In [ ]:
### to view the columns of the data set
df_genetic.columns


##### The columns names are not properly written. The code below would take care of the above output.

In [ ]:
#### To standardise the columns by making the cases lower, removing white spaces and replcing the space between words with underscore.

df_genetic.columns = df_genetic.columns.str.lower() ### to make the columns all lower case
df_genetic.columns = df_genetic.columns.str.replace(" ", "_") ## to make the the words between the columns have underscore
df_genetic.columns = df_genetic.columns.str.strip() ### to strip columns of whitespaces.

In [ ]:
### to check the result of the above code.
df_genetic.columns

In [ ]:
### to rename specific columns:

df_genetic = df_genetic.rename(columns={
    'autopsy_shows_birth_defect_(if_applicable)': 'autopsy_shows_birth_defect',
    'h/o_radiation_exposure_(x-ray)': 'ho_radiation_exposure_(x-ray)',
    'h/o_substance_abuse': 'ho_substance_abuse',
    'h/o_serious_maternal_illness': 'ho_serious_maternal_illness',
    'ho_radiation_exposure_(x-ray)':'ho_radiation_exposure',
    'heart_rate_(rates/min':'heart_rate',
    'respiratory_rate_(breaths/min)':'respiratory_rate',
   'white_blood_cell_count_(thousand_per_microliter)':'white_blood_cell_count',
     'blood_cell_count_(mcl)': 'blood_cell_count',
    'folic_acid_details_(peri-conceptional)':'folic_acid_details'
})


In [ ]:
df_genetic.columns

##### the result of the name column remove unnecessary headers like if applicable in : 'autopsy_shows_birth_defect_(if_applicable)' and the others included in the above code before the result.

In [ ]:
### To check for unique values in each columns
df_genetic.nunique()

##### The above result shows the unique value of each columns and the columns with only one variable would be dropped as they would be irrelevant for the machine learning model evaluation since machine learning requires varible for prediction in the supervised learning classifiers to be used in the task such as: Decision tree,Support vector machine and Decision tree model.

#### And the columns to be dropped are: test_1,test_2,Test_3,test_4,test_5 and parental_consent.


In [ ]:
### To drop those columns:
df_genetic.drop(["test_1", "test_2", "test_3", "test_4","test_5","parental_consent","patient_id"], axis = 1, inplace = True)

In [ ]:
df_genetic.columns

In [ ]:
#### Summary statistics of the numerical columns:
df_genetic.describe().T

##### The above result shows the mean, median and maximun values for each numerical columns.

In [ ]:
num_cols = df_genetic.select_dtypes(include = "number").columns
cat_cols = df_genetic.select_dtypes(include = "object").columns

In [ ]:
df_genetic[num_cols].corr()

##### The above result shows that no two column in the dataset has any relationship between each other.Due to the fact that most columns against each other has a correlation value of 0.00 and it it not within the correlation range of -1 to +1. Hence, no correlation.

### Exploratory Data Analysis:

##### Univariate Analysis:


In [ ]:
### Understanding the distribution of Patients' age:
# Histogram with KDE (smooth curve)
plt.figure(figsize=(8,5))
sns.histplot(df_genetic['patient_age'], bins=5, kde=True, color="skyblue")
plt.title("Distribution of Patients' Age", fontsize=14)
plt.xlabel("Age", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.show()


##### The result of the above shows that patients' age are almost evenly distributed with no particular age range being more than the other.


In [ ]:
### To understand the genetic disorder frequency in the dataset
sns.countplot(x='genetic_disorder', data=df_genetic)
plt.xticks(rotation=45)


##### The most common genetic disorder is 'Mitochondrial genetic inheritance disorder', followed by single-gene inheritance diseases in the data set.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df_genetic['patient_age'], color="skyblue")
plt.title("Boxplot of Patient Age", fontsize=14)
plt.xlabel("Age", fontsize=12)
plt.show()


##### The box plot above shows the the summary or descriptive statistics of the patient with the half of the patient being around & years and the least age being a year old and the maximum age being around 13 years old. Most importanlty, no outliers.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='gender', hue='gender', data=df_genetic, palette="Set2", legend=False)
plt.title("Distribution of Patients by Gender", fontsize=14)
plt.xlabel("Gender", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.show()

##### The geneder distribution of patients shows that higher number of patients are male followed by female then amabiguous.

In [ ]:
### Understanding the distribution of the Status column:

plt.figure(figsize=(6,4))
sns.countplot(x='status',hue = 'status', data=df_genetic, palette="Set3")
plt.title("Patient Status Distribution", fontsize=14)
plt.xlabel("Status", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=30)
plt.show()


##### The Patients status distribution shows that most of the patients with genetic disorders are alive and less are dead

##### Bivariate Analysis: relationship between two variables;

In [ ]:
sns.countplot(x='genetic_disorder', hue='gender', data=df_genetic)
plt.xticks(rotation=45)


##### The above result shows that the genedtic disorder 'Single gene inheritance disorder' has the highest number of male with female and ambigous category being the sae value,while the mitochondrial genetic inheritance disorders have the male and female gender with almost the same value count around 900. With the multifactorial genetic inheritance disorders being the least.

#### To check the relationship between folic acid intake and Birth defects

plt.figure(figsize=(8,5))
sns.countplot(x='folic_acid_details', hue='autopsy_shows_birth_defect', data=df_genetic, palette="Set2")
plt.title("Folic Acid Intake vs. Birth Defects", fontsize=14)
plt.xlabel("Folic Acid Details", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Birth Defect")
plt.show()


##### For most patients autopsy birth defect was not applicable as it dominates both groups of those who took folic acid and those who did not.
##### For those who took folic acid: NO
* the birth defects seems slightly around 510 in number comapred to those who did not have birth defects with count of 505.
##### For those who did not take folic acid:YES
* those who had birth defects are less than those who did not have any birth defects.

##### In conclusion, taking folic acid or not had no significant difference in birth defects occurence. Both groups have similiar protion of 'Yes' and 'No'


In [ ]:


plt.figure(figsize=(8,5))
sns.countplot(x='folic_acid_details', hue='birth_defects', data=df_genetic, palette="Set2")
plt.title("Folic Acid Intake vs. Birth Defects", fontsize=14)
plt.xlabel("Folic Acid Details", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Birth Defect")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(
    x='ho_radiation_exposure_(x-ray)',
    hue='autopsy_shows_birth_defect',
    data=df_genetic,
    palette="Set2"
)
plt.title("Radiation Exposure (X-ray) vs. Birth Defects", fontsize=14)
plt.xlabel("Radiation Exposure (X-ray)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Birth Defect")
plt.show()


##### For most patients autopsy birth defect was not applicable as it dominates the data set.

##### Patients  not expose to radation:No
* those who had birth defects were around 250 in number and those who did not have birth defects are 240 in number

##### Patients expose to radation:Yes
* those who had birth defects were around 240 in number and those who did not have birth defects are 230 in number.

##### In conclusion, being exposed to radiation had little no impact in birth defect occurence. Both groups have similar proportions of 'yes' and 'no'.

In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(x='assisted_conception_ivf/art', 
              hue='genetic_disorder', 
              data=df_genetic, 
              palette="Set2")

plt.title("Assisted Conception vs. Genetic Disorder", fontsize=14)
plt.xlabel("Assisted Conception (IVF/ART)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.legend(title="Genetic Disorder")
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.countplot(data=df_genetic, 
              x='ho_substance_abuse', 
              hue='genetic_disorder', 
              palette='Set2')

plt.title('Substance Exposure vs Genetic Disorder')
plt.xlabel('Substance Exposure (Yes/No)')
plt.ylabel('Count')
plt.legend(title='Genetic Disorder')
plt.show()


### Machine Learning Process:

In [ ]:
#### Data splitting and then dropping the target variable to avoid bias.
## Splitting data into x and y
x =  df_genetic.drop(["status"], axis = 1)
y = df_genetic["status"].replace({"Deceased": 0, "Alive": 1}) #### encoding the categorical variable to 0 and 1

In [ ]:
## checking count of each class and to know which category has majority class to help understand 
### whether the cataegory class is balanced which might favour a particular category in the model.
y.value_counts()  


In [ ]:
### Using undersampling to help with the imbalance data of the category class.
### the undersampler randomly removes sample from majority class to help balance the data.

sampler = RandomUnderSampler()
resampled_x, resampled_y = sampler.fit_resample(x,y)

In [ ]:
resampled_y.value_counts()

In [ ]:
## creating pipelines using defined functions:
def processor(resampled_x, resampled_y):
  x_train, x_test, y_train, y_test = train_test_split(resampled_x, resampled_y, test_size = 0.25, random_state = 42)
  num_pipe = Pipeline([
      ("imputer", SimpleImputer(strategy = "median")),
      ("scaler", StandardScaler())
  ])

  cat_pipe = Pipeline([
      ("imputer", SimpleImputer(strategy = "most_frequent")),
      ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
  ])

  preprocessor = ColumnTransformer([
      ("num", num_pipe, resampled_x.select_dtypes(include=['number']).columns),
      ("cat", cat_pipe, resampled_x.select_dtypes(include=['object']).columns)
  ],
  remainder='passthrough'
  )
  return preprocessor

processor = processor(resampled_x, resampled_y)

In [ ]:
processor

#### Support Vector Machine classifier:

In [ ]:
## SUPPORT VECTOR MACHINE ALGORITHM
svm_model = Pipeline([
    ("preprocessor", processor),
    ("svm", SVC(random_state = 42))
])

svm_model.fit(x_train, y_train)

svm_prediction = svm_model.predict(x_test)

print(classification_report(y_test, svm_prediction))

print(confusion_matrix(y_test, svm_prediction))

##### the result for the Support vector model shows that the the 'deceased class 0', of all the times the model said dead 53% of the time it was correct(precision), and of the times the model actaully captured all the deceased was 65% misssing out on 35% of the deceased cases and the model balance between the recall( the actual capture) and the precision(how many deceased were gotten) was 59% slightly above 505. which means the F1 score  shows a weak balance and it is better for dead prediction than alive status

##### Therefore, the precision for the 'alive class 1', of all the times the model has false postives for the precision and 48% of the prediction which is less than 50% is slightly.While the actual capture alive cases was 35% which means 65% of actual cases were missed and the F1 score, the balance between the recall and precision is less than 50%. 

##### In conclusion, this model is not good at predicting alive class status. The true negatives and True positives are 381 and 182 respectively.


#### Random Forest Classifier:

In [ ]:
 ## RANDOM FOREST CLASSIFICATION
 rf_model = Pipeline([
    ("preprocessor", processor),
    ("rf", RandomForestClassifier(random_state = 42))
])

rf_model.fit(x_train, y_train)

rf_prediction = rf_model.predict(x_test)

print(classification_report(y_test, rf_prediction))

print(confusion_matrix(y_test, rf_prediction))

##### the result for the random forest classifier model shows that the precision, accuracy and recall are 100% which means the model is very good at capturing the true positives and true negatives both for the alive and dead class status.

##### Decision Tree Classifier:

In [ ]:
## DECISION TREE CLASSIFICATION
dt_model = Pipeline([
    ("preprocessor", processor),
    ("dt", DecisionTreeClassifier(random_state = 42))
])

dt_model.fit(x_train, y_train)

dt_prediction = dt_model.predict(x_test)

print(classification_report(y_test, dt_prediction))

print(confusion_matrix(y_test, dt_prediction))

##### the result for the Decision Tree  model shows that the precision, accuracy and recall are 100% which means the model is very good at capturing the true positives and true negatives both for the alive and dead class status.

##### The best model on performance metrics are decision tree and random forest classifier because the f1-score, recall and precision are 100%

#### Recommendations on Models:

##### The models: Decision Trees and Random Forest can be better improved as there might be some case of overfitting. To improve this, there needs to be a re-balance between the train and test data set. This is due to the 100% precision and recall value.

##### For the SVM model this can be improved by fine-tuning or readjusting some parameters to make the model balanced in predicting both the death and alive staus well. aditionally, removing some columns which are irrelevant and might be causing some noise to the model could help improve model performance.

##### Instead of relying on only one time train- test split model, suggesting to use k-validation model to train or resampling the split technique process during modeling.

##### Feature importance analysis would help to know or linit the most important variable to the model to at least five.

##### Addition of the AUC-ROC measure metrics in addition to the Precision and Recall.

#### Limitation of dataset:

##### Too much columns in the dataset perhaps dropping more columns even after dropping some initially might help the dataset perform well.